# Assignment 01

In [21]:
import pandas as pd
import os
import numpy as np

| Attribute | Detail |
|------------|---------|
| **Dataset Name** | Combined Flights 2019 (part of the larger Flight Delay Dataset 2018–2022) |
| **Source** | [Kaggle: Flight Delay Dataset 2018–2022](https://www.kaggle.com/datasets) |
| **Data Description** | This dataset contains detailed operational and performance information for all commercial flights within the United States during the specified time frame. It includes flight identifiers, origin and destination airports, unique carrier codes, scheduled and actual departure/arrival times, and critical metrics such as flight distance, elapsed time, and indicators for delays, cancellations, and diversions. |
| **Frequency** | Individual Flight Events (Daily): Each row represents a single flight, meaning the data is recorded at the highest possible granularity (per event) but is aggregated and released daily. |
| **Time Span** | Start Date: January 1, 2019<br>End Date: December 31, 2019 |
| **Task Note** | While individual flight cancellation is a classification task, we will remake the task to predict the amount of canceled and delayed flights on a given day/at a given airport, turning this into a time series forecasting problem. |


## Check for Missing Values

In [2]:
def check_missing_values(df: pd.DataFrame, df_name: str = "DataFrame"):

    if df.empty:
        print(f"\nWarning: {df_name} is empty. Cannot perform missing value check.")
        return

    
    separator_length = 60
    print("\n" + "="*separator_length)
   
    print(f"## Detailed Missing Value Check in {df_name}")
    print("="*separator_length)

   
    missing_values = df.isnull().sum()
    
    missing_values = missing_values[missing_values > 0]

    if missing_values.empty:
        print(f"No missing (NaN) values found in the {df_name}.")
    else:
        print(f"Missing values found for the following variables in {df_name}:")
        print("\nColumn Name | NaN Count")
        print("--------------------|----------")
        
        for column, count in missing_values.sort_values(ascending=False).items():
            print(f"{column.ljust(18)}| {count}")

    print("="*separator_length)## Check for Duplicates

## Check for Duplicates

In [3]:
def check_for_duplicates(df: pd.DataFrame, df_name: str, key: list):
  
    separator_length = 60
    print("\n" + "="*separator_length)
    print(f"## Duplicate Entry Check for: {df_name}")
    print(f"Key used: {key}")
    print("="*separator_length)

    if df.empty:
        print(f"Warning: {df_name} is empty. Skipping duplicate check.")
        return

    duplicate_rows = df[df.duplicated(subset=key, keep=False)]

    if duplicate_rows.empty:
        print(f"No duplicate entries found for the key {tuple(key)} in {df_name}.")
    else:
        num_duplicates = len(duplicate_rows)
       
        num_groups = num_duplicates - duplicate_rows.drop_duplicates(subset=key).shape[0]
        
        print(f"{num_duplicates} duplicate rows identified based on the key {tuple(key)}.")
        print(f"(These rows belong to {num_groups} unique sets of duplicate key values.)")
        
        print("\nSample of Duplicate Rows (showing all entries for the duplicated key):")
      
        print(duplicate_rows.sort_values(by=key).head())
        
    print("="*separator_length)

## Check continuity

In [4]:
def validate_time_series_continuity(df: pd.DataFrame, df_name: str, date_col: str, route_key: list):

    separator_length = 80
    print("\n" + "="*separator_length)
    print("## Validate the Timestamp Sequence (Daily Continuity)")
    print("="*separator_length)

    min_date = df[date_col].min().normalize()
    max_date = df[date_col].max().normalize()

    expected_range = pd.date_range(start=min_date, end=max_date, freq='D')
    total_days_expected = len(expected_range)

    print(f"Time Span Covered: {min_date.date()} to {max_date.date()}")
    print(f"Total Unique Days Expected: {total_days_expected}")

    unique_days = df[date_col].dt.normalize().nunique()
    print(f"Total Unique Days Found: {unique_days}")

    if unique_days == total_days_expected:
        print("All days in the time span are present in the dataset (at least one flight).")
    else:
        missing_dates = expected_range[~expected_range.isin(df[date_col].dt.normalize().unique())]
        print(f"Warning: Missing day numbers in the overall time series. Expected {total_days_expected}, found {unique_days}.")
        print(f"Missing Dates Sample (First 5): {missing_dates[:5].dt.date.tolist()}")
        
    df['DateOnly'] = df[date_col].dt.normalize()

    pivot_table = df.pivot_table(
        index=route_key, 
        columns='DateOnly', 
        values='DailyCanceledAndDelayedTotal',
        aggfunc='first' # Take the first aggregated value for the day-route combination
    )

   
    missing_days_per_series = pivot_table.isnull().sum(axis=1)
    series_with_gaps = missing_days_per_series[missing_days_per_series > 0]

    print(f"\nTotal unique flight routes (time series): {len(pivot_table)}")
    print(f"Number of routes with missing days (internal gaps): {len(series_with_gaps)}")

    if len(series_with_gaps) == 0:
        print("All unique flight routes have data for every single day in the span.")
    else:
        print(f"Significant gaps found. {len(series_with_gaps)} out of {len(pivot_table)} routes have missing days.")
        print("\nSample routes with missing days (count of missing days):")
        print(series_with_gaps.head())
        print("\nNote: This is expected! Flights are rarely daily. The gaps indicate days a specific route had no operations.")
        

    df.drop(columns=['DateOnly'], inplace=True)
    print("="*80)
    
    return df

## Feature Engineeringdef create_daily_delay_cancellation_feature(df: pd.DataFrame) -> pd.DataFrame:


In [5]:
def create_daily_delay_cancellation_feature(df: pd.DataFrame) -> pd.DataFrame:

    is_delayed_indicator = (df['ArrDelay'].fillna(0) >= 15).astype(np.int8)
    is_cancelled_indicator = df['Cancelled'].astype(np.int8)
    
    df['TotalDailySignificantDelays'] = (
        is_delayed_indicator.groupby(df['FlightDate']).transform('sum')
    )
    
    df['TotalDailyCanceled'] = (
        is_cancelled_indicator.groupby(df['FlightDate']).transform('sum')
    )
    
    print("New features 'TotalDailyCanceled' and 'TotalDailySignificantDelays' created successfully using transform.")
    return df

## Plot

In [ ]:
def plot_daily_trends_by_year(df: pd.DataFrame, years: range):

    required_cols = ['FlightDate', 'Year', 'TotalDailyCanceled', 'TotalDailySignificantDelays']
    if df.empty or not all(col in df.columns for col in required_cols):
        print("\nSkipping Daily Trends visualization: DataFrame is empty or required columns are missing.")
        return

    print("\n" + "="*80)
    print("## 5. Visualizing Daily Event Counts by Year (2018-2022)")
    print("="*80)
    
    daily_df = df.drop_duplicates(subset=['FlightDate']).copy()

    for year in years:
        df_year = daily_df[daily_df['Year'] == year].sort_values('FlightDate')
        
        if df_year.empty:
            print(f"No data found for year {year}. Skipping plots.")
            continue
            

        plt.figure(figsize=(14, 6))
        sns.lineplot(
            data=df_year, 
            x='FlightDate', 
            y='TotalDailySignificantDelays', 
            label='Significant Delays (>= 15 min)',
            color='coral',
            linewidth=1.5
        )
        plt.title(f'Total Daily Significant Delays (>= 15 min) in {year}')
        plt.xlabel('Date')
        plt.ylabel('Total Daily Count')
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.show()

      
        plt.figure(figsize=(14, 6))
        sns.lineplot(
            data=df_year, 
            x='FlightDate', 
            y='TotalDailyCanceled', 
            label='Cancellations',
            color='dodgerblue',
            linewidth=1.5
        )
        plt.title(f'Total Daily Cancellations in {year}')
        plt.xlabel('Date')
        plt.ylabel('Total Daily Count')
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.show()
        
    print("\nGenerated daily line plots for Total Daily Significant Delays and Total Daily Cancellations for each year.")


## Dataset: Flight Delays and Cancellations

### Load dataset + merge + check duplicates

In [6]:
print("--- Starting Data Loading and Validation Process (Kaggle Notebook) ---")

data_dir = '/kaggle/input/flight-delay-dataset-20182022/'

all_dfs = []
flight_df = pd.DataFrame()

try:
    print("\nAttempting to find and load flight data from the mounted Kaggle input directory...")

    if os.path.exists(data_dir):
        all_files_in_dir = os.listdir(data_dir)
        files_to_load = sorted([f for f in all_files_in_dir if f.startswith('Combined') and f.endswith('.csv')])
        
        if not files_to_load:
            print(f"Error: Found directory, but no CSV files in {data_dir}.")
    else:
        print(f"Error: Kaggle input directory not found: {data_dir}.")
        files_to_load = []  

    for filename in files_to_load:
        filepath = os.path.join(data_dir, filename)
        print(f"Loading {filename}...")

        df_year = pd.read_csv(filepath)

        df_year['FlightDate'] = pd.to_datetime(df_year['FlightDate'])
        all_dfs.append(df_year)

    if all_dfs:
        flight_df = pd.concat(all_dfs, ignore_index=True)
        print("Data loaded and concatenated successfully! ")
        print(f"Combined Flight data shape: {flight_df.shape}")
        print(f"Date range: {flight_df['FlightDate'].min()} to {flight_df['FlightDate'].max()}")
        print("\nSample of Loaded Flight Data:")
        print(flight_df.head())
    else:
        print("Error: No flight data files were found or loaded.")

except Exception as e:
    print(f"An unexpected error occurred: {e}")

all_dfs = []

--- Starting Data Loading and Validation Process (Kaggle Notebook) ---

Attempting to find and load flight data from the mounted Kaggle input directory...
Loading Combined_Flights_2018.csv...
Loading Combined_Flights_2019.csv...
Loading Combined_Flights_2020.csv...
Loading Combined_Flights_2021.csv...
Loading Combined_Flights_2022.csv...
Data loaded and concatenated successfully! 
Combined Flight data shape: (29193782, 61)
Date range: 2018-01-01 00:00:00 to 2022-07-31 00:00:00

Sample of Loaded Flight Data:
  FlightDate            Airline Origin Dest  Cancelled  Diverted  CRSDepTime  \
0 2018-01-23  Endeavor Air Inc.    ABY  ATL      False     False        1202   
1 2018-01-24  Endeavor Air Inc.    ABY  ATL      False     False        1202   
2 2018-01-25  Endeavor Air Inc.    ABY  ATL      False     False        1202   
3 2018-01-26  Endeavor Air Inc.    ABY  ATL      False     False        1202   
4 2018-01-27  Endeavor Air Inc.    ABY  ATL      False     False        1400   

   Dep

### Check for Missing Values

In [7]:
df_name = "Combined Flights 2018-2022" 

In [8]:

check_missing_values(flight_df, df_name) 
            


## Detailed Missing Value Check in Combined Flights 2018-2022
Missing values found for the following variables in Combined Flights 2018-2022:

Column Name | NaN Count
--------------------|----------
AirTime           | 852561
ArrDelayMinutes   | 846183
ArrivalDelayGroups| 846183
ArrDelay          | 846183
ArrDel15          | 846183
ActualElapsedTime | 845637
TaxiIn            | 793143
WheelsOn          | 793133
ArrTime           | 786177
TaxiOut           | 780561
WheelsOff         | 780551
DepDel15          | 763084
DepDelay          | 763084
DepDelayMinutes   | 763084
DepartureDelayGroups| 763084
DepTime           | 761652
Tail_Number       | 267613
DivAirportLandings| 90
CRSElapsedTime    | 22


### Check duplicate

In [16]:
print(flight_df.columns)

Index(['FlightDate', 'Airline', 'Origin', 'Dest', 'Cancelled', 'Diverted',
       'CRSDepTime', 'DepTime', 'DepDelayMinutes', 'DepDelay', 'ArrTime',
       'ArrDelayMinutes', 'AirTime', 'CRSElapsedTime', 'ActualElapsedTime',
       'Distance', 'Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek',
       'Marketing_Airline_Network', 'Operated_or_Branded_Code_Share_Partners',
       'DOT_ID_Marketing_Airline', 'IATA_Code_Marketing_Airline',
       'Flight_Number_Marketing_Airline', 'Operating_Airline',
       'DOT_ID_Operating_Airline', 'IATA_Code_Operating_Airline',
       'Tail_Number', 'Flight_Number_Operating_Airline', 'OriginAirportID',
       'OriginAirportSeqID', 'OriginCityMarketID', 'OriginCityName',
       'OriginState', 'OriginStateFips', 'OriginStateName', 'OriginWac',
       'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID', 'DestCityName',
       'DestState', 'DestStateFips', 'DestStateName', 'DestWac', 'DepDel15',
       'DepartureDelayGroups', 'DepTimeBlk', 'TaxiOu

In [18]:
DUPLICATE_KEY = ['FlightDate', 'Airline', 'Flight_Number_Operating_Airline', 'Origin', 'Dest']
check_for_duplicates(flight_df, df_name, DUPLICATE_KEY)




## Duplicate Entry Check for: Combined Flights 2018-2022
Key used: ['FlightDate', 'Airline', 'Flight_Number_Operating_Airline', 'Origin', 'Dest']
22 duplicate rows identified based on the key ('FlightDate', 'Airline', 'Flight_Number_Operating_Airline', 'Origin', 'Dest').
(These rows belong to 11 unique sets of duplicate key values.)

Sample of Duplicate Rows (showing all entries for the duplicated key):
       FlightDate                                   Airline Origin Dest  \
290695 2018-01-05                      Delta Air Lines Inc.    JFK  ATL   
290696 2018-01-05                      Delta Air Lines Inc.    JFK  ATL   
248826 2018-01-07                     SkyWest Airlines Inc.    RDM  MFR   
250657 2018-01-07                     SkyWest Airlines Inc.    RDM  MFR   
226686 2018-01-10  GoJet Airlines, LLC d/b/a United Express    MSP  STL   

        Cancelled  Diverted  CRSDepTime  DepTime  DepDelayMinutes  DepDelay  \
290695       True     False        1540      NaN              

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


### Validate the Timestamp Sequence:

In [19]:
if not flight_df.empty and 'FlightDate' in flight_df.columns:
    print("\n" + "="*80)
    print("## Validate the Timestamp Sequence (Daily Continuity)")
    print("="*80)
    
    min_date = flight_df['FlightDate'].min()
    max_date = flight_df['FlightDate'].max()
    
    time_span = max_date - min_date
    total_days_expected = time_span.days + 1 

    print(f"Time Span Covered: {min_date.date()} to {max_date.date()}")
    print(f"Total Days Expected: {total_days_expected}")

    unique_dates = flight_df['FlightDate'].dt.date.nunique() 
    print(f"Total Unique Days Found: {unique_dates}")

    if unique_dates == total_days_expected:
        print("Timestamp sequence is continuous: All days in the span are present.")
    else:
        missing_days_count = total_days_expected - unique_dates
        print(f"Warning: Missing day(s) in the time series. Expected {total_days_expected} days, found {unique_dates} unique days.")
        print(f"Total of {missing_days_count} day(s) are missing from the time span.")
        
    print("="*80)
else:
    print("\nSkipping timestamp sequence validation: DataFrame is empty or 'FlightDate' column is missing.")



## Validate the Timestamp Sequence (Daily Continuity)
Time Span Covered: 2018-01-01 to 2022-07-31
Total Days Expected: 1673
Total Unique Days Found: 1673
Timestamp sequence is continuous: All days in the span are present.


### Run Feature Engineering

In [24]:
flight_df = create_daily_delay_cancellation_feature(flight_df)


print("\n" + "="*80)
print(f"## Sample Rows Showing New Aggregated Features: TotalDailyCanceled and TotalDailySignificantDelays")
print("="*80)
    
sample_cols = ['FlightDate', 'TotalDailyCanceled', 'TotalDailySignificantDelays']
    
print(flight_df[sample_cols].sort_values(by='FlightDate').head(10))

New features 'TotalDailyCanceled' and 'TotalDailySignificantDelays' created successfully using transform.

## Sample Rows Showing New Aggregated Features: TotalDailyCanceled and TotalDailySignificantDelays
       FlightDate  TotalDailyCanceled  TotalDailySignificantDelays
264031 2018-01-01                 129                         3962
246003 2018-01-01                 129                         3962
246002 2018-01-01                 129                         3962
262304 2018-01-01                 129                         3962
262305 2018-01-01                 129                         3962
246001 2018-01-01                 129                         3962
246000 2018-01-01                 129                         3962
245999 2018-01-01                 129                         3962
245998 2018-01-01                 129                         3962
245997 2018-01-01                 129                         3962


### Plot

In [ ]:
plot_daily_trends_by_year(flight_df, years=range(2018, 2023))